# 01 - Profile the World Bank RTFP Kenya file
Run after placing the CSV in `data/landing/`. Record every answer in `docs/source_profile.md`.

In [ ]:
from pathlib import Path

import pandas as pd

path = sorted(Path("../data/landing").glob("KEN_RTFP_mkt_2007_*.csv"))[-1]  # or archive/
df = pd.read_csv(path)
print(path.name, df.shape)  # the 2026-08-24 release was 55,224 x 46

## Structure and grain

In [ ]:
print(df.dtypes.to_string())
print("max rows per (geo_id, DATES):", df.groupby(["geo_id", "DATES"]).size().max())  # must be 1
print(df["DATES"].min(), "->", df["DATES"].max())

## Geography, currency, components

In [ ]:
print(df[["currency"]].drop_duplicates())
print(
    "distinct adm1 / adm2 / markets / geo_id:",
    df.adm1_name.nunique(),
    df.adm2_name.nunique(),
    df.mkt_name.nunique(),
    df.geo_id.nunique(),
)
print(sorted(df.adm1_name.dropna().unique()))  # spelling variants? ~47 counties expected
print(df["components"].value_counts().head())

## Which columns vary over time vs by market only?

In [ ]:
for c in [
    "start_dense_data",
    "last_survey_point",
    "data_coverage",
    "data_coverage_recent",
    "index_confidence_score",
    "spatially_interpolated",
]:
    print(c, "| max distinct per market:", df.groupby("geo_id")[c].nunique().max())

## Per-product distributions (units, ranges, inflation scale, trust range)

In [ ]:
products = sorted(c[2:] for c in df.columns if c.startswith("c_"))
print(products)
for p in products:
    print("\n==", p)
    print(
        df[[f"o_{p}", f"h_{p}", f"l_{p}", f"c_{p}", f"inflation_{p}", f"trust_{p}"]]
        .describe()
        .T[["count", "min", "50%", "max"]]
    )

## The bare product columns (`beans`, `maize_fao`, `livestock_goat_s_fao`) - what do they hold?

In [ ]:
print(df[["beans", "maize_fao", "livestock_goat_s_fao"]].describe().T)
print(df["beans"].value_counts(dropna=False).head())

## OHLC ordering and interpolation share over time

In [ ]:
p = "maize_fao"
bad = ~((df[f"l_{p}"] <= df[f"c_{p}"]) & (df[f"c_{p}"] <= df[f"h_{p}"]))
print("close outside [low, high]:", bad.mean())
print(df.groupby("year")["spatially_interpolated"].mean())

## Inflation definition test
Compare supplied `inflation_maize_fao` with close / close 12 months earlier for one market. Is it a YoY percent or a fraction? Check the Documentation PDF too.

In [ ]:
m = df[df.geo_id == df.geo_id.iloc[0]].sort_values("DATES")
m = m.assign(yoy_calc=(m["c_maize_fao"] / m["c_maize_fao"].shift(12) - 1) * 100)
m[["DATES", "c_maize_fao", "inflation_maize_fao", "yoy_calc"]].tail(15)